### Set up Experiment 1: CoT without giving answer vs CoT giving answer

This notebook reads in 4600 balanced examples from the dataset (across all 4 streets of poker), to use as our base prompt testset.

Then, it modifies to examples be used for comparing CoT without giving answer vs CoT giving answer.

### Select balanced dataset by poker stage
Create a 4600-example dataset: 600 preflop, 1000 flop, 1500 turn, 1500 river

This distribution is chosen to focus on the more complex later stages of poker (turn and river), while still including a reasonable number of preflop and flop examples.


In [4]:
#!/usr/bin/env python3
# url: https://huggingface.co/datasets/RZ412/PokerBench

from datasets import load_dataset
import json

# load and sample
dataset = load_dataset("RZ412/PokerBench")
ds20000 = dataset["train"].shuffle(seed=42).select(range(20000))

# Convert to a list of dicts and write a proper JSON array (not JSON Lines)
records = list(ds20000)

def get_stage(prompt):
    """
    Determine poker stage from prompt:
    - Preflop: no stage markers
    - Flop: contains "flop c"
    - Turn: contains "turn c"
    - River: contains "river c"
    """
    if "river c" in prompt:
        return "river"
    elif "turn c" in prompt:
        return "turn"
    elif "flop c" in prompt:
        return "flop"
    else:
        return "preflop"

# Categorize all prompts by stage
stage_prompts = {"preflop": [], "flop": [], "turn": [], "river": []}

for prompt in records:
    instruction = prompt["instruction"]
    stage = get_stage(instruction)
    stage_prompts[stage].append(prompt)

# Print counts
for stage, prompts in stage_prompts.items():
    print(f"{stage.capitalize()}: {len(prompts)} prompts")

# Select balanced dataset: 400 preflop, 800 flop, 1000 turn, 1000 river
balanced_prompts = {
    "preflop": stage_prompts["preflop"][:600],
    "flop": stage_prompts["flop"][:1000],
    "turn": stage_prompts["turn"][:1500],
    "river": stage_prompts["river"][:1500],
}

print(f"\nTotal selected: {sum(len(v) for v in balanced_prompts.values())} prompts")

# Save balanced dataset
with open("datasets/pokerbench_balanced_4600.json", "w", encoding="utf-8") as f:
    json.dump(balanced_prompts, f, ensure_ascii=False, indent=2)

print("Saved to datasets/pokerbench_balanced_4600.json")

Preflop: 2265 prompts
Flop: 1097 prompts
Turn: 7313 prompts
River: 9325 prompts

Total selected: 4600 prompts
Saved to datasets/pokerbench_balanced_4600.json


### Inject CoT instructions into Poker Prompts
Generate cot_no_answer and cot_with_answer datasets

In [5]:

import json

with open("datasets/pokerbench_balanced_4600.json","r",encoding="utf-8") as f:
    data = json.load(f)

do_not_explain_str = "Decide on an action based on the strength of your hand on this board, your position, and actions before you. Do not explain your answer.\nYour optimal action is:"
cot_str = """Explain your reasoning step by step. Format your answer in this structure:
1. Stage: <Preflop / Flop / Turn / River>
2. Known info: <board cards, hero hand, stack sizes, position>
3. Opponent range estimate: <brief logic>
4. Pot odds and/or equity estimate if applicable: <numbers or qualitative>
5. Action reasoning: <why fold / call / raise>

Then, print "FINAL DECISION ::: "

Finally, provide your final decision as one of the following exact formats:
<FOLD / CHECK / CALL / BET X / RAISE X / ALL IN>"""
no_cot_str = """
Decide on an action based on the strength of your hand on this board, your position, and actions before you. Do NOT explain your answer.

Print "FINAL DECISION ::: "

Then, give the optimal action as one of the following exact formats:
<FOLD / CHECK / CALL / BET X / RAISE X / ALL IN>"""

def replace_instructions(instruction, output=None, cot=True):
    if not output:
        return instruction.replace(do_not_explain_str, cot_str if cot else no_cot_str)
    else:
        optimal_answer_str = f"Your optimal action is: {output}\n\n"
        return instruction.replace(do_not_explain_str, optimal_answer_str + cot_str)


# Pick 60, 80, 100, 100 from preflop, flop, turn, river respectively
num_per_stage = {
    "preflop": 600,
    "flop": 1000,
    "turn": 1500,
    "river": 1500,
}
direct_answer = []
cot_no_ans = []
cot_with_ans = []
for stage in ["preflop", "flop", "turn", "river"]:
    stage_data = data[stage][:num_per_stage[stage]]
    direct_answer.extend([replace_instructions(example["instruction"], None, cot=False) for example in stage_data])
    cot_no_ans.extend([replace_instructions(example["instruction"], None) for example in stage_data])
    cot_with_ans.extend([replace_instructions(example["instruction"], example["output"]) for example in stage_data])

with open("datasets/pokerbench_direct_answer.json", "w", encoding="utf-8") as f:
    json.dump(direct_answer, f, ensure_ascii=False, indent=2)
    
with open("datasets/pokerbench_cot_no_answer.json", "w", encoding="utf-8") as f:
    json.dump(cot_no_ans, f, ensure_ascii=False, indent=2)

with open("datasets/pokerbench_cot_with_answer.json", "w", encoding="utf-8") as f:
    json.dump(cot_with_ans, f, ensure_ascii=False, indent=2)

print("Saved CoT datasets.")

Saved CoT datasets.


### Optional: generate a short sample for manual testing

In [3]:
# extract 15 cot examples for testing, put into a txt for copy and paste

number_to_extract = 30
with open("datasets/pokerbench_cot_no_ans_short.txt", "w", encoding="utf-8") as f:
    for example in cot_no_ans[:number_to_extract]:
        f.write(example + "\n----------------------------------------\n")
with open("datasets/pokerbench_cot_with_ans_short.txt", "w", encoding="utf-8") as f:
    for example in cot_with_ans[:number_to_extract]:
        f.write(example + "\n----------------------------------------\n")